In [1]:
import gzip
import numpy as np
import pandas as pd


In [2]:
# HLA-DQA1 is not listed because it consistently didn't pass QC in all datasets typed with HLA-HD
classical_class_I_II = {'HLA-A', 'HLA-B', 'HLA-C', 'HLA-DPA1', 'HLA-DPB1', 'HLA-DQB1', 'HLA-DRA', 'HLA-DRB1'}

# MACs
mac_thresholds = [5, 10, 20, 50]

In [6]:
cov_tsv_10perc_cases = {
    'ALL': '../data/1000G/GWAS_binary/ALL/seed_20_ratio_0.1_cov.tsv.gz',
    'AFR': '../data/1000G/GWAS_binary/AFR/seed_21_ratio_0.1_cov.tsv.gz',
    'AMR': '../data/1000G/GWAS_binary/AMR/seed_22_ratio_0.1_cov.tsv.gz',
    'EAS': '../data/1000G/GWAS_binary/EAS/seed_23_ratio_0.1_cov.tsv.gz',
    'EUR': '../data/1000G/GWAS_binary/EUR/seed_24_ratio_0.1_cov.tsv.gz',
    'SAS': '../data/1000G/GWAS_binary/SAS/seed_25_ratio_0.1_cov.tsv.gz'
}

cov_tsv_20perc_cases = {
    'ALL': '../data/1000G/GWAS_binary/ALL/seed_20_ratio_0.2_cov.tsv.gz',
    'AFR': '../data/1000G/GWAS_binary/AFR/seed_21_ratio_0.2_cov.tsv.gz',
    'AMR': '../data/1000G/GWAS_binary/AMR/seed_22_ratio_0.2_cov.tsv.gz',
    'EAS': '../data/1000G/GWAS_binary/EAS/seed_23_ratio_0.2_cov.tsv.gz',
    'EUR': '../data/1000G/GWAS_binary/EUR/seed_24_ratio_0.2_cov.tsv.gz',
    'SAS': '../data/1000G/GWAS_binary/SAS/seed_25_ratio_0.2_cov.tsv.gz'
}

cov_tsv_30perc_cases = {
    'ALL': '../data/1000G/GWAS_binary/ALL/seed_20_ratio_0.3_cov.tsv.gz',
    'AFR': '../data/1000G/GWAS_binary/AFR/seed_21_ratio_0.3_cov.tsv.gz',
    'AMR': '../data/1000G/GWAS_binary/AMR/seed_22_ratio_0.3_cov.tsv.gz',
    'EAS': '../data/1000G/GWAS_binary/EAS/seed_23_ratio_0.3_cov.tsv.gz',
    'EUR': '../data/1000G/GWAS_binary/EUR/seed_24_ratio_0.3_cov.tsv.gz',
    'SAS': '../data/1000G/GWAS_binary/SAS/seed_25_ratio_0.3_cov.tsv.gz'
}

nocov_tsv_10perc_cases = {
    'ALL': '../data/1000G/GWAS_binary/ALL/seed_20_ratio_0.1_nocov.tsv.gz',
    'AFR': '../data/1000G/GWAS_binary/AFR/seed_21_ratio_0.1_nocov.tsv.gz',
    'AMR': '../data/1000G/GWAS_binary/AMR/seed_22_ratio_0.1_nocov.tsv.gz',
    'EAS': '../data/1000G/GWAS_binary/EAS/seed_23_ratio_0.1_nocov.tsv.gz',
    'EUR': '../data/1000G/GWAS_binary/EUR/seed_24_ratio_0.1_nocov.tsv.gz',
    'SAS': '../data/1000G/GWAS_binary/SAS/seed_25_ratio_0.1_nocov.tsv.gz'
}

nocov_tsv_20perc_cases = {
    'ALL': '../data/1000G/GWAS_binary/ALL/seed_20_ratio_0.2_nocov.tsv.gz',
    'AFR': '../data/1000G/GWAS_binary/AFR/seed_21_ratio_0.2_nocov.tsv.gz',
    'AMR': '../data/1000G/GWAS_binary/AMR/seed_22_ratio_0.2_nocov.tsv.gz',
    'EAS': '../data/1000G/GWAS_binary/EAS/seed_23_ratio_0.2_nocov.tsv.gz',
    'EUR': '../data/1000G/GWAS_binary/EUR/seed_24_ratio_0.2_nocov.tsv.gz',
    'SAS': '../data/1000G/GWAS_binary/SAS/seed_25_ratio_0.2_nocov.tsv.gz'
}



In [7]:
def read_pvalue_tsv(filepath):
    df = pd.read_csv(filename, 
                     usecols = ['output', 'pheno', 'allele', 'ac', 'mac', 'p', 'error'], 
                     dtype = {'output': 'string', 'pheno': 'string', 'allele': 'string', 'ac': 'Int32', 'mac': 'Int32', 'p': 'float32', 'error': 'string'}, 
                     sep = '\t')
    
    df['gene'] = df['allele'].apply(lambda x: x.split('*')[0])
    df['phenotype'] = df.output + '_' + df.pheno
    df = df.drop(columns = ['output', 'pheno'])

    df = df[df.gene.isin(classical_class_I_II)]

    # Sanity info:
    n_phenotypes = len(df.phenotype.unique())
    df['error_proportion'] = df.groupby('phenotype')['error'].transform(lambda x: x.notna().mean())
    print(f'{n_phenotypes:,} tested phenotypes')
    print(f'{len(df[df["error_proportion"] > 0].phenotype.unique())} ({len(df[df["error_proportion"] > 0].phenotype.unique()) / n_phenotypes * 100:.3f}%) phenotypes had at least one numerical problem in regression')
    print(f'{len(df[df["error_proportion"] > 0.1].phenotype.unique())} ({len(df[df["error_proportion"] > 0.1].phenotype.unique()) / n_phenotypes * 100:.3f}%) phenotypes had numerical problem in regression in more than 10% of alleles')
    print(f'{len(df[df["error_proportion"] > 0.5].phenotype.unique())} ({len(df[df["error_proportion"] > 0.5].phenotype.unique()) / n_phenotypes * 100:.3f}%) phenotypes had numerical problem in regression in more than 50% of alleles')
    print(f'{len(df[df["error_proportion"] > 0.9].phenotype.unique())} ({len(df[df["error_proportion"] > 0.9].phenotype.unique()) / n_phenotypes * 100:.3f}%) phenotypes had numerical problem in regression in more than 90% of alleles')

    result = {}
    for mac in mac_thresholds:
        df_temp = df[df.mac >= mac]
    
        # Get all unique alleles and make sure that they all had the same AC values
        df_alleles = df_temp.groupby('allele')[['ac']].nunique().reset_index()
        assert len(df_alleles[df_alleles.ac != 1]) == 0 
        n_alleles = len(df_alleles)

        df_min_pvalues = df_temp[['phenotype', 'p']].groupby('phenotype').min().reset_index()
        min_pvalues_q5 = np.nanquantile(df_min_pvalues.p, 0.05)
        m_eff = 0.05 / min_pvalues_q5

        result[f'MAC{mac}_n_alleles'] = f'{n_alleles}'
        result[f'MAC{mac}_m_eff'] = f'{m_eff:.0f}'
        result[f'MAC{mac}_m_eff_perc'] = f'{m_eff / n_alleles * 100:.0f}'
    return result


In [8]:
results = []
for label in ['ALL', 'AFR', 'AMR', 'EAS', 'EUR', 'SAS']:
    print(label)
    filename = cov_tsv_10perc_cases[label]
    result = read_pvalue_tsv(filename)
    result['Dataset'] = label
    result['% Cases'] = 10
    results.append(result)

for label in ['ALL', 'AFR', 'AMR', 'EAS', 'EUR', 'SAS']:
    print(label)
    filename = cov_tsv_20perc_cases[label]
    result = read_pvalue_tsv(filename)
    result['Dataset'] = label
    result['% Cases'] = 20
    results.append(result)

for label in ['ALL', 'AFR', 'AMR', 'EAS', 'EUR', 'SAS']:
    print(label)
    filename = cov_tsv_30perc_cases[label]
    result = read_pvalue_tsv(filename)
    result['Dataset'] = label
    result['% Cases'] = 30
    results.append(result)

ALL
30,000 tested phenotypes
0 (0.000%) phenotypes had at least one numerical problem in regression
0 (0.000%) phenotypes had numerical problem in regression in more than 10% of alleles
0 (0.000%) phenotypes had numerical problem in regression in more than 50% of alleles
0 (0.000%) phenotypes had numerical problem in regression in more than 90% of alleles
AFR
30,000 tested phenotypes
151 (0.503%) phenotypes had at least one numerical problem in regression
1 (0.003%) phenotypes had numerical problem in regression in more than 10% of alleles
0 (0.000%) phenotypes had numerical problem in regression in more than 50% of alleles
0 (0.000%) phenotypes had numerical problem in regression in more than 90% of alleles
AMR
30,000 tested phenotypes
0 (0.000%) phenotypes had at least one numerical problem in regression
0 (0.000%) phenotypes had numerical problem in regression in more than 10% of alleles
0 (0.000%) phenotypes had numerical problem in regression in more than 50% of alleles
0 (0.000%)

In [9]:
pd.DataFrame(results)

,MAC5_n_alleles,MAC5_m_eff,MAC5_m_eff_perc,MAC10_n_alleles,MAC10_m_eff,MAC10_m_eff_perc,MAC20_n_alleles,MAC20_m_eff,MAC20_m_eff_perc,MAC50_n_alleles,MAC50_m_eff,MAC50_m_eff_perc,Dataset,% Cases
0,283,223,79,236,187,79,196,159,81,143,124,87,ALL,10
1,148,108,73,128,95,74,103,78,75,62,53,85,AFR,10
2,153,94,62,109,70,64,64,42,65,21,18,84,AMR,10
3,135,93,69,110,74,67,84,59,70,44,33,76,EAS,10
4,133,90,68,105,71,68,77,54,70,43,33,76,EUR,10
5,138,87,63,107,72,67,77,52,67,46,33,72,SAS,10
6,283,211,75,236,188,80,196,167,85,143,132,92,ALL,20
7,148,108,73,128,96,75,103,81,79,62,54,87,AFR,20
8,153,97,63,109,75,69,64,48,75,21,18,87,AMR,20
9,135,92,68,110,77,70,84,63,75,44,36,82,EAS,20
